# The language classroom: readability, vocabulary, cloze tests and quizzes

**Course**: Artificial Intelligence (*Intelligence artificielle*) · Master 2 (English studies, Didactics) · Université Yahia Farès, Médéa
**Chapter 2**: Reading and processing text files, applied to language teaching
**Duration**: about 1 h 30

### What you will do
- Measure the **readability** of a text with a formula (Flesch for English, Kandel and Moles for French), and see what this formula cannot see.
- Draw up the **vocabulary profile** of a text: tokens, types, type/token ratio, frequent words, hapaxes.
- Make exercises: a **cloze test** with its answer key, a **comparison** between a learner's answer and a model answer, and a small multiple-choice **quiz**.

### How to use this notebook
Run each cell from top to bottom: click the ▶ button at its left, or press Shift+Enter. **You never have to write code.** Change only the fields of the cells titled "TO CHANGE".

- In Google Colab, the menu "Runtime → Run all" runs the whole notebook. The "TO CHANGE" fields appear as boxes, sliders or drop-down lists next to the code.
- In Jupyter, these fields are plain `#@param` comments: change only the value after the `=` sign.
- After changing a value, run the cell again, then the cells that follow it.

### The method: predict, run, observe, modify
Before each cell, a question asks you to **predict** the result: write down your answer. Then **run** the cell. The "What you should see" section gives the expected result and a question to help you **observe** and interpret. Finally, you **modify** a value to check that you have understood.

### The texts
Public-domain extracts: an Aesop fable in George Fyler Townsend's translation (1867), Wilde (1888), Dickens (1843), and two speeches for the mini-project (Lincoln 1863, Roosevelt 1933). The "simple text" and the learners' answers are **invented** for the example (fictitious data).

## 0. Prepare the tools
This cell loads the tools the notebook needs: `re` to split a text into words and sentences, `Counter` to count, `difflib` to compare two texts, `random` for the quiz draws, `pandas` for tables and `matplotlib` for the chart. They are all already installed in Colab: nothing to download.

**Before running:** what do you think this cell will display? A long list, a table, or almost nothing?

In [ ]:
#@title Prepare the tools (run this first)
import re                          # regular expressions: split into words and sentences
import random                      # random draws (quiz), reproducible thanks to a seed
import difflib                     # compare two texts
import io                          # read a string as if it were a file
from collections import Counter    # count frequencies
import pandas as pd                # tables
import matplotlib.pyplot as plt    # charts
pd.set_option("display.width", 140)
print("Tools ready.")

### What you should see
```
Tools ready.
```
The `import` lines display nothing; only the last line, a `print()`, writes this message.

**Observe:** a number has appeared at the left of the cell. What do you think would happen if you ran a cell further down without running this one first?

## 1. The readability of a text
A language teacher is always choosing texts: is this one within reach of my class? In 1948, Rudolf Flesch proposed a numerical answer for English: a text is harder when its **sentences are long** and when its **words are long**, measured in syllables. In 1958, L. Kandel and A. Moles adapted his formula to French. These formulas do not read the text: they count.

The next cell stores the four texts of the notebook in a dictionary (chapter 2): the key is the name of the text, the value is the text itself. It then displays the number of words in each one.

**Before running:** rank these four texts from the easiest to the hardest for B1 learners: simple text, Aesop, Wilde, Dickens. Write down your ranking: you will compare it with the formula's ranking.

In [ ]:
#@title The four texts of this notebook (public domain, except the simple text, invented)
texts = {
    "A simple text (invented)": """Sam has a little dog. His name is Max. Max is brown and white.
In the morning, Max sleeps on the bed. In the afternoon, he plays in the garden with a red ball.
In the evening, Sam gives him some water. Max drinks fast. Then he sits on Sam's knees.
He closes his eyes. Sam reads a book. The boy and the dog are happy.""",
    "Aesop, The Hare and the Tortoise (tr. Townsend, 1867)": """A Hare one day ridiculed the short feet and slow pace of the Tortoise, who replied, laughing: "Though you be swift as the wind, I will beat you in a race." The Hare, believing her assertion to be simply impossible, assented to the proposal; and they agreed that the Fox should choose the course and fix the goal. On the day appointed for the race the two started together. The Tortoise never for a moment stopped, but went on with a slow but steady pace straight to the end of the course. The Hare, lying down by the wayside, fell fast asleep. At last waking up, and moving as fast as he could, he saw the Tortoise had reached the goal, and was comfortably dozing after her fatigue. Slow but steady wins the race.""",
    "Wilde, The Happy Prince (1888)": """High above the city, on a tall column, stood the statue of the Happy Prince. He was gilded all over with thin leaves of fine gold, for eyes he had two bright sapphires, and a large red ruby glowed on his sword-hilt. He was very much admired indeed. "He is as beautiful as a weathercock," remarked one of the Town Councillors who wished to gain a reputation for having artistic tastes; "only not quite so useful," he added, fearing lest people should think him unpractical, which he really was not. "Why can't you be like the Happy Prince?" asked a sensible mother of her little boy who was crying for the moon. "The Happy Prince never dreams of crying for anything." "I am glad there is some one in the world who is quite happy," muttered a disappointed man as he gazed at the wonderful statue.""",
    "Dickens, A Christmas Carol (1843)": """Marley was dead: to begin with. There is no doubt whatever about that. The register of his burial was signed by the clergyman, the clerk, the undertaker, and the chief mourner. Scrooge signed it: and Scrooge's name was good upon 'Change, for anything he chose to put his hand to. Old Marley was as dead as a door-nail. Mind! I don't mean to say that I know, of my own knowledge, what there is particularly dead about a door-nail. I might have been inclined, myself, to regard a coffin-nail as the deadest piece of ironmongery in the trade. But the wisdom of our ancestors is in the simile; and my unhallowed hands shall not disturb it, or the Country's done for.""",
}
texts_language = "en"
for name, t in texts.items():
    print(name, ":", len(re.findall(r"\w[\w'’-]*", t)), "words")

### What you should see
```
A simple text (invented) : 66 words
Aesop, The Hare and the Tortoise (tr. Townsend, 1867) : 134 words
Wilde, The Happy Prince (1888) : 146 words
Dickens, A Christmas Carol (1843) : 121 words
```
Here, a "word" is a sequence of letters that may contain an apostrophe or a hyphen: "can't" and "door-nail" each count as one word.

**Observe:** the Dickens extract is shorter than the Wilde one. Is that a reason for it to be easier?

### TO CHANGE: the text to study
This cell is a **form**. In Colab, `choice` is a drop-down list that offers the four texts; `my_text` is a box where you can paste your own text (an extract from a textbook, an anonymous piece of learner writing); `language` gives the language of the text, `en` or `fr`. If `my_text` is not empty, it is the text that is studied. The rest of the notebook works on this text.

**Before running:** the text chosen by default is Wilde's. With which words will the display begin?

In [ ]:
#@title TO CHANGE: the text to study
choice = "Wilde, The Happy Prince (1888)" #@param ["A simple text (invented)", "Aesop, The Hare and the Tortoise (tr. Townsend, 1867)", "Wilde, The Happy Prince (1888)", "Dickens, A Christmas Carol (1843)"]
my_text = "" #@param {type:"string"}
language = "en" #@param ["en", "fr"]
if my_text.strip() != "":
    text = my_text                 # your own text, if there is one
else:
    text = texts[choice]           # otherwise, the text chosen in the list
print("Language:", language)
print("Start of the text:", text[:160], "...")

### What you should see
```
Language: en
Start of the text: High above the city, on a tall column, stood the statue of the Happy Prince. He was gilded all over with thin leaves of fine gold, for eyes he had two bright sa ...
```
`text[:160]` keeps the first 160 characters: the display stops in the middle of a word.

**Observe:** if you change `choice` later, which cells will you need to run again so that the results are about the new text?

### Counting syllables without a dictionary
To apply the formula, we must count the syllables of each word. The program has no pronunciation dictionary: it follows a **simple, transparent rule**.
1. Count the **groups of vowels** that follow each other: in "beautiful", "eau", "i" and "u" make three groups, so three syllables.
2. Remove a **silent final "e"** when it follows a consonant: "prince" has two groups (i, e) but one syllable.
3. Keep a final "le" after a consonant, which is pronounced (*lit-tle*, *ta-ble*).
4. Count at least one syllable per word.

(The same tool also handles French, where a final "-es" is treated like "-e".) The last line of the cell applies this rule to the first eight words of the text.

**Before running:** count the syllables of "High above the city, on a tall column" yourself, as in ordinary speech. Will the program find the same as you?

In [ ]:
#@title Count the syllables of a word (tool)
VOWELS = "aeiouyàâäéèêëîïôöùûüÿœæ"
def count_syllables(word, language):
    """Estimate: number of groups of vowels, minus a silent final "e"."""
    word = word.lower()
    n = len(re.findall(f"[{VOWELS}]+", word))      # "ea", "ou", "eau": one single group
    if language == "fr" and word.endswith("es"):
        word = word[:-1]                             # French "charmantes" is treated like "charmante"
    silent = len(word) > 1 and word.endswith("e") and word[-2] not in VOWELS
    if language == "en" and word.endswith("le") and len(word) > 2 and word[-3] not in VOWELS:
        silent = False                               # little, table: this final "le" is pronounced
    if silent:
        n = n - 1                                    # prince, whale: the final "e" is silent
    return max(n, 1)                                 # at least one syllable per word
print([(w, count_syllables(w, language)) for w in re.findall(r"\w[\w'’-]*", text)[:8]])

### What you should see
```
[('High', 1), ('above', 2), ('the', 1), ('city', 2), ('on', 1), ('a', 1), ('tall', 1), ('column', 2)]
```
On these eight words, the rule is right. "above" (three groups: a, o, e) loses its silent "e"; "High" has one group of vowels, "i" (the "gh" is not a vowel).

**Observe:** English spelling is famous for its irregularities. Think of two common words for which a rule based on spelling will surely go wrong.

### Checking the tool
A simple rule is bound to make mistakes. To find out where, we compare it with known answers: the dictionary `reference` gives, for eleven words, the real number of syllables, counted by hand (ordinary pronunciation).

**Before running:** among these eleven words (little, whale, beautiful, statue, the, people, created, makes, loved, business, 1843), which ones will trap the rule? How many right answers do you expect?

In [ ]:
#@title Check the tool on words whose answer we know
# Real number of syllables, in ordinary speech, counted by hand
reference = {"little": 2, "whale": 1, "beautiful": 3, "statue": 2, "the": 1, "people": 2,
             "created": 3, "makes": 1, "loved": 1, "business": 2, "1843": 5}
right = 0
for word, real in reference.items():
    estimate = count_syllables(word, "en")
    if estimate == real:
        right = right + 1
        print(f"{word:<10} estimated {estimate}, real {real}   right")
    else:
        print(f"{word:<10} estimated {estimate}, real {real}   ERROR")
print(right, "right out of", len(reference))

### What you should see
Six right out of eleven (`6 right out of 11`). The errors:
- "created" (2 instead of 3): "ea" is one group of vowels, but we say *cre-a-ted*;
- "makes" and "loved" (2 instead of 1): the "e" of the endings "-es" and "-ed" is often silent, and the rule only knows the silent final "e";
- "business" (3 instead of 2): the "i" is not pronounced (*busi-ness*);
- "1843" (1 instead of 5): a number contains no vowel, but we say *eigh-teen for-ty-three*.

**Observe:** these eleven words were chosen to trap the rule; they do not tell us its error rate on a real text (on the first eight words of Wilde, it made no mistake). How would you estimate its reliability honestly?

### The formula
The cell defines the function `readability(text, language)`. It splits the text into **sentences** (at the signs `.` `!` `?` `…`), then into **words**, counts the syllables and calculates two averages: the number of words per sentence and the number of syllables per word. Then comes the score:

- for English, **Flesch (1948)**: score = 206.835 − 1.015 × (words per sentence) − 84.6 × (syllables per word);
- for French, **Kandel and Moles (1958)**: score = 206.835 − 1.015 × (words per sentence) − 73.6 × (syllables per word); the constant is often rounded to 207.

The **higher** the score, the **easier** the text. Both averages are rounded before the calculation, so that you can do the sum again on a calculator.

**Before running:** are Wilde's sentences rather short (fewer than 12 words) or rather long (more than 20)? Will his score be above or below 50?

In [ ]:
#@title The readability formula (tool) and the result for your text
def readability(text, language):
    sentences = [p for p in re.split(r"[.!?…]+", text) if re.search(r"\w", p)]  # cut at . ! ? …
    words = re.findall(r"\w[\w'’-]*", text)            # "can't", "sword-hilt": one single word
    syllables = sum([count_syllables(w, language) for w in words])
    wps = round(len(words) / len(sentences), 1)         # average sentence length, in words
    spw = round(syllables / len(words), 2)              # average word length, in syllables
    if language == "en":
        score = 206.835 - 1.015 * wps - 84.6 * spw      # Flesch (1948), for English
    else:
        score = 206.835 - 1.015 * wps - 73.6 * spw      # Kandel and Moles (1958), for French
    return {"sentences": len(sentences), "words": len(words), "syllables": syllables,
            "words per sentence": wps, "syllables per word": spw, "score": round(score, 1)}
for key, value in readability(text, language).items():
    print(key, ":", value)

### What you should see
```
sentences : 8
words : 146
syllables : 209
words per sentence : 18.2
syllables per word : 1.43
score : 67.4
```
On a calculator: 206.835 − 1.015 × 18.2 − 84.6 × 1.43 = 206.835 − 18.473 − 120.978 = 67.4 (rounded).

Look at the number of sentences: 8, whereas the extract has 7. In `"Why can't you be like the Happy Prince?" asked a sensible mother…`, the question mark inside the quotation is taken for the end of a sentence. With 7 sentences, the average would be 20.9 words and the score 64.6.

**Observe:** which of the two terms of the formula takes the most points away from the score: sentence length or word length?

### Interpreting and comparing
Flesch linked his scores to levels, calibrated on American pupils whose first language is English. The same scale is often used for French, for want of anything better:

| Score | Level | Readers Flesch had in mind (American school) |
|---|---|---|
| 90 and above | very easy | age 10-11 |
| 80 to 90 | easy | age 11-12 |
| 70 to 80 | fairly easy | age 12-13 |
| 60 to 70 | standard | age 13-15 |
| 50 to 60 | fairly difficult | age 15-18 |
| 30 to 50 | difficult | college students |
| below 30 | very difficult | university graduates |

These bands are **approximate**: they say nothing about foreign-language learners, nor about the A1 to C2 levels of the Common European Framework of Reference for Languages. The cell defines `interpret(score)`, which turns a score into a level, then calculates the readability of the four texts in a `pandas` table, sorted from the easiest to the hardest.

**Before running:** look again at your ranking from the start. Will the formula agree with you?

In [ ]:
#@title Interpret the scores and compare the four texts
BANDS = [(90, "very easy"), (80, "easy"), (70, "fairly easy"), (60, "standard"),
         (50, "fairly difficult"), (30, "difficult")]
def interpret(score):
    """Turns a score into a level, on Flesch's scale (also used for French)."""
    for threshold, level in BANDS:
        if score >= threshold:
            return level
    return "very difficult"
table = pd.DataFrame([readability(t, texts_language) for t in texts.values()], index=list(texts))
table["level"] = [interpret(s) for s in table["score"]]
print(table.sort_values("score", ascending=False).to_string())

### What you should see
```
                                                       sentences  words  syllables  words per sentence  syllables per word  score        level
A simple text (invented)                                      11     66         77                 6.0                1.17  101.8    very easy
Aesop, The Hare and the Tortoise (tr. Townsend, 1867)          7    134        177                19.1                1.32   75.8  fairly easy
Dickens, A Christmas Carol (1843)                              9    121        170                13.4                1.40   74.8  fairly easy
Wilde, The Happy Prince (1888)                                 8    146        209                18.2                1.43   67.4     standard
```
Three remarks:
- The simple text goes above 100: the scale has no ceiling, 100 is not a maximum mark.
- Aesop's fable, a story for children, comes out "fairly easy", just ahead of Dickens: its sentences are long (19.1 words) but its words are short. Yet Townsend's Victorian vocabulary (*ridiculed*, *assented*, *wayside*) is far from easy for a learner.
- Dickens gets a good score thanks to short sentences such as "Marley was dead: to begin with." and "Mind!", which counts as a sentence of one word.

**Observe:** is "the wisdom of our ancestors is in the simile" hard for a learner because of the length of the sentence, or for another reason that the formula does not see?

### The scores as a picture
A horizontal bar chart, one bar per text. The thin grey vertical lines mark the limits between the levels (30, 50, 60, 70, 80, 90).

**Before running:** which text will have the longest bar? Which one the shortest?

In [ ]:
#@title The scores as a picture
ordered = table.sort_values("score")
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.barh(ordered.index, ordered["score"], height=0.5, color="#2a78d6")
for y, s in enumerate(ordered["score"]):
    ax.text(s + 1.5, y, f"{s}  ({interpret(s)})", va="center", fontsize=9)
for threshold, level in BANDS:
    ax.axvline(threshold, color="#dddddd", linewidth=1, zorder=0)  # limits between the levels
ax.set_xlim(min(0, ordered["score"].min() - 5), 125)
ax.set_xlabel("Readability score (the higher it is, the easier the text)")
ax.set_title("Readability of the four texts (Flesch formula)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

### What you should see
Four blue bars, from the highest score (at the top) to the lowest (at the bottom): simple text 101.8 (very easy), Aesop 75.8 and Dickens 74.8 (fairly easy), Wilde 67.4 (standard). Each bar carries its score and its level.

**Observe:** the chart gives an impression of precision (a bar, a number with one decimal place). What caption would you write under it so that a colleague does not take it for an exact measurement of difficulty?

### Why a different formula for French?
The two formulas differ by one coefficient only: 84.6 for English, 73.6 for French. French words have more syllables on average than English words; with the English coefficient, every French text would look too hard. This cell applies both formulas to the chosen text.

**Before running:** with the French formula, will Wilde's score be higher or lower than 67.4? By how much, roughly?

In [ ]:
#@title Both formulas on the same text
r = readability(text, language)
flesch = 206.835 - 1.015 * r["words per sentence"] - 84.6 * r["syllables per word"]
kandel_moles = 206.835 - 1.015 * r["words per sentence"] - 73.6 * r["syllables per word"]
print("Flesch formula (English)           :", round(flesch, 1), "->", interpret(flesch))
print("Kandel and Moles formula (French)  :", round(kandel_moles, 1), "->", interpret(kandel_moles))
print("Gap:", round(kandel_moles - flesch, 1), "points, that is 11 x", r["syllables per word"], "syllables per word")

### What you should see
```
Flesch formula (English)           : 67.4 -> standard
Kandel and Moles formula (French)  : 83.1 -> easy
Gap: 15.7 points, that is 11 x 1.43 syllables per word
```
The same text moves from "standard" to "easy" depending on the formula. The gap is exactly (84.6 − 73.6) × 1.43 = 11 × 1.43 = 15.7 points.

**Observe:** a score only makes sense compared with other scores obtained with the same formula, in the same language. Can we say that an English text scoring 70 is "easier" than a French text scoring 65?

## 2. The vocabulary profile of a text
As in chapter 2, we distinguish **tokens** (*occurrences*), each appearance of a word, and **types** (*formes*), the different words. Their ratio, the **TTR** (*type/token ratio*), is a rough measure of the variety of the vocabulary: close to 1, almost every word is new; lower, the text repeats the same words more often.

The splitting is the one from chapter 2, `re.findall(r"\w+", …)` on the lowercased text: "Sam's" gives two tokens, "sam" and "s". So there are a few more tokens than words in part 1.

**Before running:** roughly how many different types does Wilde's text contain? More or less than half of its tokens?

In [ ]:
#@title Tokens, types and vocabulary richness
tokens = re.findall(r"\w+", text.lower())       # as in chapter 2: "Sam's" gives sam + s
types = set(tokens)                              # each type is kept only once
print("Tokens (occurrences):", len(tokens))
print("Types (different words):", len(types))
print("Type/token ratio (TTR):", round(len(types) / len(tokens), 2))

### What you should see
```
Tokens (occurrences): 148
Types (different words): 102
Type/token ratio (TTR): 0.69
```
148 tokens, not 146: "can't" gives "can" and "t", "sword-hilt" gives "sword" and "hilt". 102 types for 148 tokens: more than two thirds.

**Observe:** which words do you think come back most often in this passage? Write down three guesses before the cell on frequencies.

### Careful: the TTR depends on length
The longer a text is, the more it reuses words already seen (articles, pronouns, prepositions): its TTR drops mechanically. To compare two texts, you must therefore compare them **at equal length**, for example on their first 50 words. The cell calculates both TTRs for each of the four texts.

**Before running:** the simple text repeats "Sam" and "Max". Will it have the lowest TTR of the four?

In [ ]:
#@title The TTR depends on the length of the text
rows = []
for name, t in texts.items():
    toks = re.findall(r"\w+", t.lower())
    rows.append({"text": name, "tokens": len(toks),
                 "TTR (whole text)": round(len(set(toks)) / len(toks), 2),
                 "TTR (first 50 words)": round(len(set(toks[:50])) / 50, 2)})
print(pd.DataFrame(rows).to_string(index=False))

### What you should see
```
                                                 text  tokens  TTR (whole text)  TTR (first 50 words)
                             A simple text (invented)      67              0.64                  0.68
Aesop, The Hare and the Tortoise (tr. Townsend, 1867)     134              0.61                  0.78
                       Wilde, The Happy Prince (1888)     148              0.69                  0.82
                    Dickens, A Christmas Carol (1843)     127              0.64                  0.78
```
On the whole texts, the simple text (0.64) looks as varied as Dickens (0.64) and more varied than Aesop (0.61): this is an effect of its shortness (67 tokens). On the first 50 words, the difference appears clearly: 0.68 for the simple text, against 0.78 to 0.82 for the three literary texts.

**Observe:** to compare the lexical richness of two learners' essays, one of 80 words and the other of 300, what would you do before calculating their TTR?

### The most frequent words
`Counter` counts each type. Without a filter, the top places go to **stop words** (*mots vides*): articles, prepositions, pronouns, auxiliaries. The cell defines a list of stop words, then counts again without them. This list will also be used for the cloze test and the quiz.

**Before running:** look again at your three guesses. What do you think is the most frequent word in the passage, stop words included?

In [ ]:
#@title The most frequent words, with and without stop words
stop_words = {"the", "a", "an", "and", "or", "but", "nor", "of", "to", "in", "on", "at", "by", "for",
              "with", "from", "as", "into", "that", "this", "which", "who", "what", "there", "he",
              "she", "it", "they", "we", "you", "i", "him", "her", "them", "his", "its", "their", "my",
              "me", "our", "your", "is", "was", "were", "are", "be", "been", "had", "has", "have", "do",
              "don", "not", "no", "so", "if", "than", "then", "very", "all", "one", "some", "s", "t",
              "can", "could", "would", "should", "will", "shall", "upon", "about", "up", "out",
              "am", "only", "quite", "much", "more", "too"}
freq = Counter(tokens)
print("With stop words:   ", freq.most_common(6))
content_words = [w for w in tokens if w not in stop_words]
print("Without stop words:", Counter(content_words).most_common(6))

### What you should see
```
With stop words:    [('the', 9), ('he', 7), ('a', 6), ('of', 5), ('happy', 4), ('was', 4)]
Without stop words: [('happy', 4), ('prince', 3), ('statue', 2), ('crying', 2), ('high', 1), ('above', 1)]
```
Without the stop words, "happy" comes first: 4 occurrences, 3 of them in the name "the Happy Prince", and one in "quite happy". Words that appear the same number of times keep their order of appearance in the text.

**Observe:** the program counts "Happy" in a name and "happy" as an adjective as the same word. What does the repetition of "happy" suggest about the story, and what does the program not "understand" here?

### Hapaxes
A **hapax** (*hapax legomenon*, plural *hapaxes*) is a word that appears only once in a text. For the teacher, long hapaxes are candidates for the vocabulary to explain before reading: the learner cannot rely on meeting the word a second time to guess its meaning.

**Before running:** out of the 102 types in the passage, how many do you think are hapaxes: a quarter, half, three quarters?

In [ ]:
#@title Words used only once (hapax legomena)
hapax = [w for w, n in freq.items() if n == 1]
print(len(hapax), "hapaxes out of", len(types), "types, that is", round(100 * len(hapax) / len(types)), "%")
# Words you may need to explain before reading: long content words used once
to_explain = sorted([w for w in hapax if w not in stop_words and len(w) >= 8])
print("Hapaxes of 8 letters or more:", to_explain)

### What you should see
```
85 hapaxes out of 102 types, that is 83 %
Hapaxes of 8 letters or more: ['anything', 'artistic', 'beautiful', 'councillors', 'disappointed', 'muttered', 'remarked', 'reputation', 'sapphires', 'sensible', 'unpractical', 'weathercock', 'wonderful']
```
More than four types out of five appear only once: this is usual in a short text.

**Observe:** among these 13 words, which ones would you really explain to B1 learners? Which ones are transparent for learners who know French (*artistic*, *reputation*), and which ones are "false friends" (*sensible*)?

## 3. An automatic cloze test
The **cloze test** (*test de closure*, or *texte à trous*), proposed by Wilson Taylor in 1953, consists in removing words from a text and asking the reader to find them again. Taylor first designed it… as a measure of readability. Two classic ways to choose the words:
- remove **one word in n** (here, one *content* word in n: stop words are skipped): the exercise is rather about overall comprehension;
- remove the **long words**: the exercise is rather about vocabulary.

The function `to_remove(word, rank, mode, n, min_length)` applies the rule to one word; `rank` is the position of the word among the content words. The cell tries it on the first eight content words of the text, with n = 3 and a minimum length of 6 letters.

**Before running:** among "High, above, city, tall, column, stood, statue, Happy", which ones will be removed with the rule "one in 3"? With the rule "6 letters or more"?

In [ ]:
#@title The rule: which words to remove? (tool)
def to_remove(word, rank, mode, n, min_length):
    """True if this content word must become a gap."""
    if mode == "every n-th content word":
        return rank % n == 0                  # the n-th content word, the 2n-th, etc.
    else:
        return len(word) >= min_length        # all the words that are long enough
# Test on the first eight content words of the text (n = 3, minimum length = 6)
text_content_words = [w for w in re.findall(r"\w+", text) if w.isalpha() and w.lower() not in stop_words]
for rank, word in enumerate(text_content_words[:8], start=1):
    one_in_3 = to_remove(word, rank, "every n-th content word", 3, 6)
    long_ones = to_remove(word, rank, "long words", 3, 6)
    print(f"{rank}. {word:<12}", "one in 3:", one_in_3, "|", "6 letters or more:", long_ones)

### What you should see
```
1. High         one in 3: False | 6 letters or more: False
2. above        one in 3: False | 6 letters or more: False
3. city         one in 3: True | 6 letters or more: False
4. tall         one in 3: False | 6 letters or more: False
5. column       one in 3: False | 6 letters or more: True
6. stood        one in 3: True | 6 letters or more: False
7. statue       one in 3: False | 6 letters or more: True
8. Happy        one in 3: False | 6 letters or more: False
```
`True` means "to remove". "One in 3" removes the 3rd and the 6th content words (city, stood): `rank % n == 0` checks that the remainder of the division of `rank` by `n` is zero. "6 letters or more" removes column and statue.

**Observe:** the rule "one in 3" does not look at the word. Is this a weakness or a strength for a comprehension exercise?

### Making the cloze test
The function `cloze_test` goes through the text piece by piece: `re.findall(r"\w+|\W+", text)` splits it into words and "non-words" (spaces, punctuation), so that it can be put back together exactly. Each content word to remove is replaced by a numbered gap and stored in the answer key. The option `first_letter` leaves the first letter as a hint. The last line tries the function on a sentence by Wilde, removing the words of 6 letters or more.

**Before running:** in "He was gilded all over with thin leaves of fine gold.", which words have 6 letters or more?

In [ ]:
#@title Make a cloze test (tool)
def cloze_test(text, mode, n, min_length, first_letter):
    result = ""                                      # the gapped text, built piece by piece
    answer_key = []                                  # the removed words, in order
    rank = 0                                         # number of the current content word
    for piece in re.findall(r"\w+|\W+", text):       # words and "non-words" (spaces, punctuation)
        if piece.isalpha() and piece.lower() not in stop_words:
            rank = rank + 1
            if to_remove(piece, rank, mode, n, min_length):
                answer_key.append(piece)
                hint = piece[0] if first_letter else ""      # first letter, if the option is ticked
                piece = f"({len(answer_key)}) {hint}________"   # the word becomes a gap
        result = result + piece
    return result, answer_key
print(cloze_test("He was gilded all over with thin leaves of fine gold.", "long words", 3, 6, False))

### What you should see
```
('He was (1) ________ all over with thin (2) ________ of fine gold.', ['gilded', 'leaves'])
```
The function returns two things at once (a *tuple*, chapter 2): the gapped text and the list of removed words. The punctuation is kept.

**Observe:** for this exercise, would "gold" have been a better gap than "leaves"? What do we lose when the machine chooses the words by their length?

### TO CHANGE: settings of the cloze test
Choose the `mode` ("every n-th content word" or "long words"), the value of `n`, the minimum length `min_length` (used only in "long words" mode) and the option `first_letter`. The cell displays the gapped text, the answer key, a **word bank** in alphabetical order (for an easier version, in differentiated teaching) and the number of gaps.

**Before running:** with one content word in 4, in a passage of 148 tokens with many stop words, how many gaps do you expect?

In [ ]:
#@title TO CHANGE: settings of the cloze test
mode = "every n-th content word" #@param ["every n-th content word", "long words"]
n = 4 #@param {type:"slider", min:2, max:10, step:1}
min_length = 9 #@param {type:"slider", min:5, max:12, step:1}
first_letter = False #@param {type:"boolean"}
gapped, answer_key = cloze_test(text, mode, n, min_length, first_letter)
print(gapped)
print()
print("Answer key:", ", ".join([f"({i}) {word}" for i, word in enumerate(answer_key, start=1)]))
print("Word bank (alphabetical order):", " - ".join(sorted(answer_key, key=str.lower)))
print(len(answer_key), "gaps for", len(re.findall(r"\w+", text)), "words")

### What you should see
Wilde's passage with 18 gaps; it begins: "High above the city, on a (1) ________ column, stood the statue of the (2) ________ Prince." Then:
```
Answer key: (1) tall, (2) Happy, (3) thin, (4) eyes, (5) large, (6) sword, (7) beautiful, (8) Councillors, (9) having, (10) added, (11) think, (12) like, (13) sensible, (14) crying, (15) never, (16) glad, (17) disappointed, (18) statue
Word bank (alphabetical order): added - beautiful - Councillors - crying - disappointed - eyes - glad - Happy - having - large - like - never - sensible - statue - sword - tall - thin - think
18 gaps for 148 words
```
Note gap (6): the machine removed "sword" from "sword-hilt", which becomes "(6) ________-hilt".

**Modify:** switch to "long words" mode (with `min_length` = 9), then tick `first_letter`.

**Observe:** which version would you give to beginners, which one to advanced learners? Which gaps would you correct by hand: a gap that is impossible to guess (the name "Happy"), a gap that accepts several right answers ("large", "glad")?

## 4. Comparing a learner's answer with a model answer
The `difflib` module compares two sequences and calculates a **similarity ratio** between 0 (nothing in common) and 1 (identical): twice the number of common elements, divided by the total number of elements in both sequences. Here, the elements are **words**; punctuation stays attached to the word. `difflib` describes the way from the model to the answer with four operations: `equal` (identical passage), `delete` (words removed), `insert` (words added), `replace` (words replaced). The function `compare` returns this ratio and the answer with the differences marked: `[-word-]` for a word of the model that is missing, `{+word+}` for a word written by the learner.

**Before running:** "the cat sleeps on the bed" and "the dog sleeps on the bed" have 5 words in common, out of 12 words in total. What similarity ratio do you expect?

In [ ]:
#@title Compare two answers word by word (tool)
def compare(model, answer):
    a = model.split()                                   # we compare lists of words
    b = answer.split()
    matcher = difflib.SequenceMatcher(None, a, b)
    marked = []
    for operation, i1, i2, j1, j2 in matcher.get_opcodes():
        if operation == "equal":                        # identical passage: copy it
            marked.append(" ".join(a[i1:i2]))
        if operation in ("delete", "replace"):          # words of the model missing from the answer
            marked.append("[-" + " ".join(a[i1:i2]) + "-]")
        if operation in ("insert", "replace"):          # words added or changed by the learner
            marked.append("{+" + " ".join(b[j1:j2]) + "+}")
    return round(matcher.ratio(), 2), " ".join(marked)
print(compare("the cat sleeps on the bed", "the dog sleeps on the bed"))

### What you should see
```
(0.83, 'the [-cat-] {+dog+} sleeps on the bed')
```
The calculation: 2 × 5 common words / 12 words in total = 0.83. Only one word has changed, and yet the whole meaning of the sentence changes.

**Observe:** according to this calculation, what ratio would an answer get if it said roughly the same thing with other words, such as "a feline slumbers upon its couch"?

### TO CHANGE: a learner's answer
The form contains a comprehension question on Wilde's passage, a model answer and a learner's answer. **This answer is invented** for the example: it contains three typical mistakes. Replace it with a real (anonymous) answer to try the tool.

**Before running:** find the learner's mistakes in the `answer` field. Will the ratio be closer to 0.5 or to 1?

In [ ]:
#@title TO CHANGE: a learner's answer (invented answer)
question = "Why do people admire the statue?" #@param {type:"string"}
model = "People admire the statue because it is covered with gold and decorated with precious stones." #@param {type:"string"}
answer = "People admires the statue because it is cover with gold and decorated with precious stone." #@param {type:"string"}
score, marked = compare(model, answer)
print("Question:", question)
print("Similarity:", score, "that is", round(100 * score), "%")
print("Differences:", marked)
print("(Key: [-word-] = word of the model that is missing; {+word+} = word written by the learner)")

### What you should see
```
Question: Why do people admire the statue?
Similarity: 0.8 that is 80 %
Differences: People [-admire-] {+admires+} the statue because it is [-covered-] {+cover+} with gold and decorated with precious [-stones.-] {+stone.+}
(Key: [-word-] = word of the model that is missing; {+word+} = word written by the learner)
```
The three mistakes (third-person "s" with a plural subject, missing "-ed" of the passive, missing plural "s") are all found. But "stones." and "stone." are treated as two completely different words: the tool does not see that only one letter separates them.

**Observe:** would this tool save you time to correct a **dictation**? And to mark a comprehension answer?

### Similarity is not correctness
Three (**invented**) learners have answered the same question. For each one, the cell displays the similarity ratio with the model, the differences, and the judgement of a teacher (also invented) on whether the answer is right.

**Before running:** read the three answers in the code. Rank them from the best to the worst. Will the similarity ratio give the same ranking?

In [ ]:
#@title Three learners (invented answers): similarity or correctness?
answers = {
    "Learner A": ("People admires the statue because it is cover with gold and decorated with precious stone.",
                  "right meaning, three grammar mistakes"),
    "Learner B": ("Its thin gold leaves, sapphire eyes and big ruby make it look rich and splendid.",
                  "right and well paraphrased"),
    "Learner C": ("People laugh at the statue because it is covered with gold and decorated with precious stones.",
                  "wrong: complete misreading"),
}
for name, (answer_text, judgement) in answers.items():
    score, marked = compare(model, answer_text)
    print(f"{name}: similarity {score}   |   teacher's judgement (invented): {judgement}")
    print("    ", marked)

### What you should see
```
Learner A: similarity 0.8   |   teacher's judgement (invented): right meaning, three grammar mistakes
Learner B: similarity 0.13   |   teacher's judgement (invented): right and well paraphrased
Learner C: similarity 0.9   |   teacher's judgement (invented): wrong: complete misreading
```
(each line is followed by the answer with its differences marked).

The machine's ranking is the reverse of the teacher's. C, who **completely misreads** the text ("laugh at"), gets the best ratio, 0.9; B, who has **understood and paraphrased** in their own words, gets the worst, 0.13. The ratio measures surface resemblance to the model, not correctness: **similarity is not correctness**. It can help to prepare the correction of a dictation or to spot identical copies; it must never mark an open answer on its own.

**Observe:** what kind of answer does this ratio always favour? What effect would it have on learners to announce that they will be marked this way?

## 5. A small vocabulary quiz
The quiz is a multiple-choice exercise: in a sentence of the text, one word is removed, and the learner finds it among four options. This first cell prepares the ingredients:
- the **candidate words**: 5 letters or more, and no stop words;
- the **usable sentences**: those that contain at least one candidate; in each one, the word to remove is the **longest** candidate.

**Before running:** in Wilde's first sentence ("High above the city, on a tall column, stood the statue of the Happy Prince"), which word will be removed?

In [ ]:
#@title The ingredients of the quiz: candidate words and usable sentences
# Candidate words: 5 letters or more, and not stop words (sorted, for a reproducible draw)
candidates = sorted(set([w for w in re.findall(r"\w+", text) if len(w) >= 5 and w.lower() not in stop_words]))
# Usable sentences: those that contain at least one candidate word
quiz_sentences = []
for p in re.split(r"[.!?…]+", text):
    words_p = [w for w in re.findall(r"\w+", p) if w in candidates]
    if words_p:
        quiz_sentences.append((p.strip(), max(words_p, key=len)))  # (the sentence, its longest word)
print(len(candidates), "candidate words, for example:", candidates[:6])
print(len(quiz_sentences), "usable sentences. The first one, with the word to remove:")
print(quiz_sentences[:1])

### What you should see
```
45 candidate words, for example: ['Councillors', 'Happy', 'Prince', 'above', 'added', 'admired']
8 usable sentences. The first one, with the word to remove:
[('High above the city, on a tall column, stood the statue of the Happy Prince', 'column')]
```
"Councillors", "Happy" and "Prince" come first because Python sorts capital letters before small letters. The last line displays a list containing the first (sentence, word to remove) pair: "column" and "statue" both have 6 letters, and the first one in the sentence wins.

**Observe:** is the longest word always the most interesting one to find? Suggest another rule.

### TO CHANGE: settings of the quiz
Choose the number of questions `num_questions` and a **seed** (`seed`, *graine* in French). The program draws at random the sentences and the three **distractors** (the wrong answers), taken from the other candidate words of the text. `random.seed(seed)` fixes the starting point of the randomness: with the same seed, you always get the same quiz, so you can print it again or share it; with another seed, you get another quiz.

**Before running:** the distractors are drawn at random. Will they be plausible (same part of speech, same number, same form as the right answer)?

In [ ]:
#@title TO CHANGE: settings of the multiple-choice quiz
num_questions = 3 #@param {type:"slider", min:1, max:6, step:1}
seed = 2026 #@param {type:"integer"}
random.seed(seed)                                    # same seed = same quiz, every time you run it
draw = random.sample(quiz_sentences, min(num_questions, len(quiz_sentences)))
quiz_key = []
for number, (sentence, target) in enumerate(draw, start=1):
    options = random.sample([c for c in candidates if c != target], 3) + [target]
    random.shuffle(options)                          # the right answer moves to a random place
    print(f"{number}. {sentence.replace(target, '________', 1)}")
    for letter, word in zip("ABCD", options):
        print(f"     {letter}) {word}")
    right_letter = "ABCD"[options.index(target)]
    quiz_key.append(f"{number}: {right_letter} ({target})")
print("\nAnswer key:", " | ".join(quiz_key))

### What you should see
```
1. He was gilded all over with thin leaves of fine gold, for eyes he had two bright ________, and a large red ruby glowed on his sword-hilt
     A) sensible
     B) sapphires
     C) anything
     D) wished
2. He was very much ________ indeed
     A) sword
     B) sapphires
     C) tastes
     D) admired
3. "Why can't you be like the Happy ________
     A) Prince
     B) anything
     C) useful
     D) asked

Answer key: 1: B (sapphires) | 2: D (admired) | 3: A (Prince)
```
Run the cell again: the quiz is identical, thanks to the seed. Many distractors give themselves away through grammar: in question 2, only "admired" can follow "He was very much"; in question 1, only "sapphires" is a plural noun that fits after "two bright". Question 3 is cut in the middle of the quotation (the question mark ends the "sentence") and asks for a word everybody knows from the title. A learner can answer correctly without knowing the word.

**Modify:** change `seed` (for example 7), then `num_questions`.

**Observe:** how many of these questions would you keep as they are for a real class? How would you choose better distractors?

## Your turn
Three modifications to try with the "TO CHANGE" fields (then run the following cells again):
1. In the choice of text, take **Aesop**, then run parts 1 to 5 again. Which words do the cloze test and the quiz choose? Are they the words a teacher would pick in a fable?
2. Paste into `my_text` a paragraph from a textbook you use (or an anonymous piece of learner writing) and compare its score with those of the four texts. Then paste a French text and choose `language = "fr"`: the Kandel and Moles formula then applies (the list of stop words, however, remains the English one).
3. Prepare two versions of the same cloze test: one for beginners ("long words", `first_letter` ticked, with the word bank) and one for advanced learners ("every n-th content word" with n = 5, without the word bank). What makes one easier than the other?

## For the mini-project
These tools apply directly to **subject 3** (political speeches): comparing the readability and lexical richness of speeches by several speakers, periods or languages. They are also useful for **subject 1**, to describe the length and vocabulary of readers' comments.

Prepare a CSV file (in a spreadsheet: "Save as", CSV format, UTF-8 encoding) with two columns, `title` and `text`, one text per row. In Colab, tick `upload` then run the cell: a window asks you to choose the file (`files.upload()`), which `pandas.read_csv` turns into a table. Outside Colab, this function does not exist: the `try`/`except` block then displays a message instead of an error. Without a file, the cell uses two public-domain extracts from speeches (Lincoln 1863, Roosevelt 1933).

**Before running:** `upload` is not ticked. Which table will the cell use?

In [ ]:
#@title (Optional) Your own texts: a CSV file with the columns "title" and "text"
upload = False #@param {type:"boolean"}
example = """title,text
"Lincoln, Gettysburg Address (1863)","Four score and seven years ago our fathers brought forth on this continent, a new nation, conceived in Liberty, and dedicated to the proposition that all men are created equal. Now we are engaged in a great civil war, testing whether that nation, or any nation so conceived and so dedicated, can long endure. We are met on a great battle-field of that war."
"Roosevelt, First Inaugural Address (1933)","This great Nation will endure as it has endured, will revive and will prosper. So, first of all, let me assert my firm belief that the only thing we have to fear is fear itself—nameless, unreasoning, unjustified terror which paralyzes needed efforts to convert retreat into advance."
"""
corpus = pd.read_csv(io.StringIO(example))          # by default: the two extracts above
if upload:
    try:
        from google.colab import files
        uploaded = files.upload()                   # a window opens: choose your .csv file
        corpus = pd.read_csv(list(uploaded)[0])
    except ImportError:
        print("Outside Colab: put the file next to the notebook and write corpus = pd.read_csv('my_file.csv')")
print(corpus["title"].tolist())

### What you should see
```
['Lincoln, Gettysburg Address (1863)', 'Roosevelt, First Inaugural Address (1933)']
```
The titles of the two extracts included in the notebook. With your own file, your titles are displayed.

**Observe:** Lincoln's extract contains commas. Why must it be surrounded by quotation marks in the CSV file?

The cell applies the readability formula and the TTR calculation to each text in the file, and gathers the results in a table. It uses the `language` chosen in part 1.

**Before running:** Roosevelt's extract contains the famous sentence "the only thing we have to fear is fear itself". Will it be judged easier or harder than Lincoln's?

In [ ]:
#@title Readability and vocabulary of each text in the file
rows = []
for title, content in zip(corpus["title"], corpus["text"]):
    r = readability(content, language)
    toks = re.findall(r"\w+", content.lower())
    rows.append({"text": title, "words per sentence": r["words per sentence"], "score": r["score"],
                 "level": interpret(r["score"]), "TTR": round(len(set(toks)) / len(toks), 2)})
print(pd.DataFrame(rows).to_string(index=False))

### What you should see
```
                                     text  words per sentence  score            level  TTR
       Lincoln, Gettysburg Address (1863)                21.3   55.8 fairly difficult 0.72
Roosevelt, First Inaugural Address (1933)                24.0   48.8        difficult 0.92
```
Roosevelt comes out "difficult" (48.8), because of a long second sentence full of long words (*unreasoning*, *unjustified*, *paralyzes*). Yet his most famous phrase is made of short, simple words. The TTR of 0.92 is high mostly because the extract is short (see part 2).

**Observe:** a speech is made to be heard. Which limits of the readability formula will you need to discuss in your report?

## To hand in
Answer in writing, **in English**, in a few lines for each question:
1. Does the formula's ranking match yours (start of part 1)? For a text where you disagree, explain what the formula does not see: rare vocabulary, syntax, cultural references, the language of another period.
2. A colleague wants to mark learners' answers automatically with the similarity ratio. Write them a reasoned opinion (8 to 10 lines), based on learners A, B and C.
3. Choose a text for a specific class (level, aim) and build a short activity with the tools of this notebook: a readability score with a comment, three words to explain, a cloze test, two quiz questions. Say what you corrected by hand in what the machine produced, and why.

## Key points to remember
- A readability formula (Flesch, Kandel and Moles) only measures the **length of sentences and words**: it is a rough indication, useful to compare texts in the same language with the same formula, never a verdict on the real difficulty for your learners.
- The **vocabulary profile** rests on simple counts: tokens, types, TTR (to be compared at equal length), frequent words, hapaxes. The numbers point things out; reading interprets them.
- The machine **quickly makes** exercises (cloze tests, quizzes) and measures resemblance; the teacher keeps the choice of words, the quality of the distractors and the judgement on meaning: **similarity is not correctness**.

Matching lesson on the course website: [Reading and processing text files](https://progremer04.github.io/canva-au-for-ang-/en/#c2-fichiers) (tokens and types, stop words, frequencies). For the mini-project: [subject 3, political speeches](https://progremer04.github.io/canva-au-for-ang-/en/#mp-sujet-3).